# Mini Projeto:
## Preços de Combustíveis no Brasil

In [21]:
# Importando bibliotecas
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib


In [22]:
# Formula para leitura de DataFrames .CSV
df_2024_1 = pd.read_csv('Preços semestrais - AUTOMOTIVOS_2024.01.csv', sep=';', decimal=',')
df_2024_2 = pd.read_csv("Preços semestrais - AUTOMOTIVOS_2024.02.csv", sep=';', decimal=',')
df_2025_1 = pd.read_csv("Preços semestrais - AUTOMOTIVOS_2025.01.csv", sep=';', decimal=',')
df_2025_2 = pd.read_csv("Preços semestrais - AUTOMOTIVOS_2025.02.csv", sep=';', decimal=',')


In [23]:
# Contatenando (juntando todas as infomaçes) os quatro DataFrame em um.
df = pd.concat([df_2024_1, df_2024_2, df_2025_1, df_2025_2],
 ignore_index=True) # ignore_index=True dentro de um pd.concat() ele serve para reorganizar a numeração das linhas/sintese após juntar DataFrames.
df.shape 
df.head()


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL-220,1901,NaN,SANTA ESMERALDA,57312-025,GASOLINA,01/01/2024,5.68,NaN,R$ / litro,VIBRA
1,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL-220,1901,NaN,SANTA ESMERALDA,57312-025,GASOLINA ADITIVADA,01/01/2024,5.68,NaN,R$ / litro,VIBRA
2,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL-220,1901,NaN,SANTA ESMERALDA,57312-025,DIESEL S10,01/01/2024,5.93,NaN,R$ / litro,VIBRA
3,NE,AL,ARAPIRACA,COMERCIAL DE COMBUSTIVEIS E LUBRIFICANTES VITA...,02.817.655/0001-82,RODOVIA AL-220,1901,NaN,SANTA ESMERALDA,57312-025,ETANOL,01/01/2024,3.99,NaN,R$ / litro,VIBRA
4,NE,AL,ARAPIRACA,AUTO POSTO SAO FRANCISCO LTDA,12.706.529/0001-04,RUA SAO FRANCISCO,274,NaN,CENTRO,57300-080,GASOLINA,01/01/2024,5.48,NaN,R$ / litro,VIBRA


# 2 - Gasolina C no Rio de Janeiro
## O diretor quer entender como se comportaram os preços médios da Gasolina C no estado do Rio de Janeiro entre 2024 e 2025. Calcule a média do Valor de Venda por município e apresente os resultados do maior para o menor. Quais municípios praticaram os maiores preços médios no período

In [24]:
df_rj = df.loc[(df["Estado - Sigla"] == "RJ") & (df["Produto"] == "GASOLINA")]

df_municipio_venda_rj = df_rj.groupby("Municipio")["Valor de Venda"].mean().reset_index().sort_values(by="Valor de Venda", ascending=False).round(2)


In [25]:
array_venda_rj = np.array(df_municipio_venda_rj["Valor de Venda"])

media = np.mean(array_venda_rj)
mediana = np.median(array_venda_rj)

distancia = (media - mediana) / mediana

print(f"Média: R$ {media}\nMediana: R$ {mediana}")
print(f"Distância entre média e mediana: {distancia:.2%}")

print("\nOs municipios que praticaram os maiores preços médios foram:")
df_municipio_venda_rj.head()

Média: R$ 6.03625
Mediana: R$ 5.975
Distância entre média e mediana: 1.03%

Os municipios que praticaram os maiores preços médios foram:


,Municipio,Valor de Venda
29,TRES RIOS,6.59
0,ANGRA DOS REIS,6.52
18,PETROPOLIS,6.48
2,BARRA DO PIRAI,6.38
11,MACAE,6.36


# 3 - Outliers de preço: Gasolina C no RJ
## Ainda com foco na Gasolina C no Rio de Janeiro, identifique os postos com preços estatisticamente atípicos. Calcule o IQR e os limites para o Valor de Venda. Existem postos praticando preços fora do padrão? Apresente os grupos de outliers superiores e inferiores ordenados de forma adequada. Se o limite inferior for negativo, o que isso indica?

In [53]:
df_postos_rj = df_rj.groupby("Revenda")["Valor de Venda"].mean().reset_index()

array_postos_rj = np.array(df_postos_rj["Valor de Venda"])

q1 = np.percentile(array_postos_rj, 25)
q3 = np.percentile(array_postos_rj, 75)
iqr = q3 - q1

limite_inferior = q1 - (1.5 * iqr)
limite_superior = q3 + (1.5 * iqr)

outliers_inferiores = df_rj.loc[df_rj["Valor de Venda"] < limite_inferior].sort_values(by="Valor de Venda")
outliers_superiores = df_rj.loc[df_rj["Valor de Venda"] > limite_superior].sort_values(by="Valor de Venda", ascending=False)

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Limite Inferior: R$ {limite_inferior}")
print(f"Limite Superior: R$ {limite_superior}")


print("Outliers Inferiores:")
display(outliers_inferiores[["Municipio", "Revenda", "Valor de Venda"]])

print("\nOutliers Superiores:")
display(outliers_superiores[["Municipio", "Revenda", "Valor de Venda"]]) # IA DISSE PARA USAR F{LEN(OUTLIERS_SUPERIORES)}. FAZ SENTIDO?

print("\n\nSe o limite inferior for negativo, indica que todos os dados estão acima do limite.")

Q1: 5.692154135338346
Q3: 6.19
IQR: 0.49784586466165415
Limite Inferior: R$ 4.945385338345865
Limite Superior: R$ 6.936768796992482
Outliers Inferiores:


,Municipio,Revenda,Valor de Venda
48157,RIO DE JANEIRO,POSTO DE GASOLINA JARDIM SULACAP LTDA,4.84
79893,RIO DE JANEIRO,POSTO DE GASOLINA JARDIM SULACAP LTDA,4.84



Outliers Superiores:


,Municipio,Revenda,Valor de Venda
564996,RIO DE JANEIRO,POSTO JARDIM OCEANICO LTDA.,7.79
677178,RIO DE JANEIRO,POSTO JARDIM OCEANICO LTDA.,7.79
724765,RIO DE JANEIRO,POSTO JARDIM OCEANICO LTDA.,7.79
695386,RIO DE JANEIRO,POSTO JARDIM OCEANICO LTDA.,7.79
708760,RIO DE JANEIRO,POSTO JARDIM OCEANICO LTDA.,7.79
...,...,...,...
1570009,SAO GONCALO,AUTO POSTO DE COMBUSTIVEIS M C 2000 DE SANTA I...,7.39
1557778,SAO GONCALO,AUTO POSTO DE COMBUSTIVEIS M C 2000 DE SANTA I...,7.39
1572618,SAO GONCALO,AUTO POSTO DE COMBUSTIVEIS M C 2000 DE SANTA I...,7.39
1561538,SAO GONCALO,AUTO POSTO DE COMBUSTIVEIS M C 2000 DE SANTA I...,7.39




Se o limite inferior for negativo, indica que todos os dados estão acima do limite.


# 4 - Comparando estados: Etanol Hidratado
## O diretor quer comparar o preço médio do Etanol Hidratado entre RJ e SP em 2024 e 2025. Filtre os dados para os dois estados e calcule a média do Valor de Venda para cada um. Em seguida, aplique o critério de outliers em cada estado separadamente. Os limites e os casos atípicos são semelhantes ou diferentes entre os dois estados?

In [54]:
filtro_rj = df.loc[(df["Estado - Sigla"] == "RJ") & (df["Produto"] == "ETANOL")]
filtro_sp = df.loc[(df["Estado - Sigla"] == "SP") & (df["Produto"] == "ETANOL")]

array_etanol_rj = np.array(filtro_rj["Valor de Venda"])
array_etanol_sp = np.array(filtro_sp["Valor de Venda"])

# INFOS RJ
media_etanol_rj = np.mean(array_etanol_rj)
mediana_etanol_rj = np.median(array_etanol_rj)
distancia_rj = (media_etanol_rj - mediana_etanol_rj) / mediana_etanol_rj

q1_rj = np.percentile (array_etanol_rj, 25)
q3_rj = np.percentile (array_etanol_rj, 75)
iqr_rj = q3_rj - q1_rj

limite_inferior_rj = q1_rj - (1.5 * iqr_rj)
limite_superior_rj = q3_rj + (1.5 * iqr_rj)

outliers_inferiores_rj = filtro_rj.loc[filtro_rj["Valor de Venda"] < limite_inferior]
outliers_superiores_rj = filtro_rj.loc[filtro_rj["Valor de Venda"] > limite_superior]

# INFOS SP
media_etanol_sp = np.mean(array_etanol_sp)
mediana_etanol_sp = np.median(array_etanol_sp)
distancia_sp =(media_etanol_sp - mediana_etanol_sp) / mediana_etanol_sp

q1_sp = np.percentile (array_etanol_sp, 25)
q3_sp = np.percentile (array_etanol_sp, 75)
iqr_sp = q3_sp - q1_sp

limite_inferior_sp = q1 - (1.5 * iqr_sp)
limite_superior_sp = q3 + (1.5 * iqr_sp)

outliers_inferiores_sp = filtro_sp.loc[filtro_sp["Valor de Venda"] < limite_inferior]
outliers_superiores_sp = filtro_sp.loc[filtro_sp["Valor de Venda"] > limite_superior]
# /////////////////////////////////////////////////////////////////////////////////////////////

print("DADOS ETANOL HIDRATADO - RIO DE JANEIRO")
print(f"Média: R$ {media_etanol_rj.round(2)}\nMediana: R$ {mediana_etanol_rj}\nDistânci entre média e medidana: {distancia_rj:2%}")
print(f"Outliers inferiores: {len(outliers_inferiores)}\nOutliers superiores: {len(outliers_superiores)}")

print("\nDADOS ETANOL HIDRATADO - SÃO PAULO")
print(f"Média: R$ {media_etanol_sp.round(2)}\nMediana: R$ {mediana_etanol_sp}\nDistânci entre média e medidana: {distancia_sp:2%}")
print(f"Outliers inferiores: {len(outliers_inferiores)}\nOutliers superiores: {len(outliers_superiores)}")

DADOS ETANOL HIDRATADO - RIO DE JANEIRO
Média: R$ 4.43
Mediana: R$ 4.39
Distânci entre média e medidana: 1.013364%
Outliers inferiores: 2
Outliers superiores: 125

DADOS ETANOL HIDRATADO - SÃO PAULO
Média: R$ 3.9
Mediana: R$ 3.89
Distânci entre média e medidana: 0.161412%
Outliers inferiores: 2
Outliers superiores: 125


# 5 - Bairros da capital: preços dentro ou fora do padrão?
## A diretoria quer entender como os preços da Gasolina C se distribuem entre os diferentes bairros da cidade do Rio de  Janeiro. Existem bairros com dados atípicos? 
## A partir do recorte abaixo

### 001 Calcule a média do Valor de Venda por bairro (coluna Bairro) e ordene do maior para o menor.

In [36]:
df_media_bairro = df_rj.groupby("Bairro")["Valor de Venda"].mean().reset_index().sort_values(by="Valor de Venda", ascending=False)
df_media_bairro

,Bairro,Valor de Venda
353,SANTA IZABEL,7.415000
249,NOSSA SENHORA DE SANTANA,6.778889
32,BAIRRO YPU,6.747273
85,CHACARA PARAISO,6.704691
171,ITANEMA (CUNHAMBEBE),6.690000
...,...,...
0,14 DE JULHO,5.315000
427,VILA RECREIO (VILA INHOMIRIM),5.292500
229,MARIA DA GRACA,5.290000
211,JARDIM SULACAP,5.226364


In [39]:
df_rj

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
122,SE,RJ,CAMPOS DOS GOYTACAZES,POSTO DE COMBUSTIVEIS CONTORNO DE CAMPOS LTDA,31.212.889/0002-85,AVENIDA NILO PECANHA,614,/822 PATIO,PQ SANTO AMARO,28030-035,GASOLINA,01/01/2024,5.99,NaN,R$ / litro,VIBRA
130,SE,RJ,CAMPOS DOS GOYTACAZES,ARARA FLUMINENSE COMERCIO DE COMBUSTIVEIS LTDA.,07.614.346/0004-71,AVENIDA ALBERTO TORRES,100,NaN,CENTRO,28010-117,GASOLINA,01/01/2024,5.99,NaN,R$ / litro,RAIZEN
131,SE,RJ,CAMPOS DOS GOYTACAZES,POSTO SAO LUCAS DE CAMPOS LTDA,04.718.173/0001-09,"RODOVIA BR-101 KM 80,5",S/N,NaN,IBITIOCA,28120-000,GASOLINA,01/01/2024,5.89,NaN,R$ / litro,RAIZEN
133,SE,RJ,CAMPOS DOS GOYTACAZES,MOTO MERCANTIL CAMPISTA S A,28.935.153/0001-40,RUA DOS GOYTACAZES,342,NaN,CENTRO,28010-460,GASOLINA,01/01/2024,5.99,NaN,R$ / litro,VIBRA
136,SE,RJ,CAMPOS DOS GOYTACAZES,G.S. NOGUEIRA COMBUSTÍVEIS LTDA,03.784.646/0001-03,RUA JOAQUIM LORENA,6,NaN,URURAÍ,28020-230,GASOLINA,01/01/2024,5.79,NaN,R$ / litro,BRANCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1622777,SE,RJ,NOVA FRIBURGO,F.MURY DESIGN PROPAGANDA E PUBLICIDADE LTDA,7707130000299,ESTRADA FRIBURGO AMPARO,15604,LP N 0130/2018 LP N 0440/2019,CHACARA PARAISO,28605-520,GASOLINA,30/12/2025,6.69,NaN,R$ / litro,VIBRA
1622806,SE,RJ,MARICA,POSTO NOVO RECREIO LTDA,24787241000352,AVENIDA MAYSA,12656,LOTE 4 A QUADRA 24,BARRA DE MARICÁ,24915-400,GASOLINA,30/12/2025,6.39,NaN,R$ / litro,BRANCA
1622972,SE,RJ,ANGRA DOS REIS,AUTO POSTO ENCRUZO DA ENSEADA LTDA,3340868000129,RUA PREFEITO JOAO GREGORIO GALINDO,S/N,LOTE 02,CAMPO BELO (CUNHAMBEBE),23932-400,GASOLINA,31/12/2025,6.69,NaN,R$ / litro,IPIRANGA
1622973,SE,RJ,TRES RIOS,AUTO LUBRIFICADORA TRIRRIENSE LTDA,28863579000135,RUA BARAO DE ENTRE RIOS,401,NaN,CENTRO,25802-315,GASOLINA,31/12/2025,6.89,NaN,R$ / litro,IPIRANGA


In [47]:
df_rj["Regiao - Sigla"].unique()

<StringArray>
['SE']
Length: 1, dtype: str

### 002 Aplique o IQR para identificar bairros com preços médios estatisticamente atípicos — tanto os superiores quanto os inferiores.

In [38]:
array_media_bairro = np.array(df_media_bairro["Valor de Venda"])

q1_media_bairro = np.percentile(array_media_bairro, 25)
q3 = np.percentile(array_media_bairro, 75)
iqr = q3 - q1 

limite_inferior_media_bairro = q1 - (1.5 * iqr)
limite_superior_media_bairro = q3 + (1.5 * iqr)

outliers_inferiores_media_bairro = df_media_bairro.loc(df_media_bairro["Valor de Venda"] < limite_inferior_media_bairro)
outliers_superiores_media_bairro = df_media_bairro.loc(df_media_bairro["Valor de Venda"] >)